import cv2
import numpy as np

def text_to_bits(text):
    """تبدیل متن به رشته بیت‌ها (UTF-8)"""
    byte_array = text.encode('utf-8')
    bits = ''.join(format(byte, '08b') for byte in byte_array)
    return bits

def embed_lsb(image_path, text, output_path):
    """
    نهان‌سازی متن در تصویر با روش LSB.
    
    Args:
        image_path: مسیر تصویر حامل
        text: متن برای پنهان‌سازی
        output_path: مسیر ذخیره تصویر خروجی
    """
    # 1. بارگذاری تصویر
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"تصویر {image_path} یافت نشد.")
    
    # 2. تبدیل متن به بیت‌ها
    bits = text_to_bits(text)
    total_bits = len(bits)
    
    # 3. محاسبه ظرفیت تصویر (تعداد پیکسل‌ها * 3 کانال)
    height, width, channels = img.shape
    capacity = height * width * channels
    
    # 4. بررسی ظرفیت
    if total_bits > capacity:
        raise ValueError(f"ظرفیت تصویر ناکافی است. نیاز: {total_bits} بیت، موجود: {capacity} بیت.")
    
    # 5. نهان‌سازی بیت‌ها در LSB
    bit_index = 0
    for row in range(height):
        for col in range(width):
            for ch in range(channels):
                if bit_index < total_bits:
                    # دریافت مقدار پیکسل فعلی
                    pixel_val = img[row, col, ch]
                    # جایگزینی LSB با بیت پیام
                    # حذف LSB قدیمی: pixel_val & 0xFE (11111110)
                    # اضافه کردن بیت جدید: | int(bits[bit_index])
                    img[row, col, ch] = (pixel_val & 0xFE) | int(bits[bit_index])
                    bit_index += 1
                else:
                    break
            if bit_index >= total_bits:
                break
        if bit_index >= total_bits:
            break
    
    # 6. ذخیره تصویر حاوی پیام
    cv2.imwrite(output_path, img)
    print(f"✅ نهان‌سازی موفق. تصویر در {output_path} ذخیره شد.")
    print(f"   تعداد بیت‌های جاسازی‌شده: {total_bits}")
    print(f"   ظرفیت استفاده‌شده: {total_bits}/{capacity} بیت")

# مثال اجرا
if __name__ == "__main__":
    embed_lsb(
        image_path="[/mnt/data/image.png"](https://storage.gapgpt.app/media/code_interpreter/d7bb0669-0d8a-4dfe-8ccb-d36d6feb06f0/image.png%22),
        text="زندگی ادامه دارد...",
        output_path="[/mnt/data/encoded_image.png"](https://storage.gapgpt.app/media/code_interpreter/d7bb0669-0d8a-4dfe-8ccb-d36d6feb06f0/encoded_image.png%22)
    )


In [ ]:
# embed_lsb.py
import cv2

def encode_lsb(image_path: str, secret_text: str, output_path: str = "encoded_image.png"):
    """
    LSB Steganography (RGB):
    - 32 بیت اول: طول پیام به بایت (uint32, big-endian)
    - سپس بیت‌های متن UTF-8
    - هر پیکسل: 3 بیت (LSB کانال‌های B,G,R در OpenCV)
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    h, w, _ = img.shape
    flat = img.reshape(-1, 3)

    data = secret_text.encode("utf-8")
    length = len(data)
    header = length.to_bytes(4, byteorder="big")  # 32-bit length

    total_bits = 32 + length * 8
    capacity_bits = h * w * 3
    if total_bits > capacity_bits:
        raise ValueError(
            f"Not enough capacity. Need {total_bits} bits, have {capacity_bits} bits."
        )

    bit_index = 0

    # Write header (length)
    for byte_val in header:
        for bit_pos in range(8):
            bit = (byte_val >> (7 - bit_pos)) & 1
            pixel_idx = bit_index // 3
            channel = bit_index % 3
            flat[pixel_idx, channel] = (flat[pixel_idx, channel] & 0xFE) | bit
            bit_index += 1

    # Write payload (UTF-8 bytes)
    for byte_val in data:
        for bit_pos in range(8):
            bit = (byte_val >> (7 - bit_pos)) & 1
            pixel_idx = bit_index // 3
            channel = bit_index % 3
            flat[pixel_idx, channel] = (flat[pixel_idx, channel] & 0xFE) | bit
            bit_index += 1

    encoded = flat.reshape(h, w, 3)
    cv2.imwrite(output_path, encoded)
    return output_path

if __name__ == "__main__":
    input_image = "input.png"
    secret_text = "زندگی ادامه دارد, حتی وقتی سخت است, حتی وقتی درد دارد, دردها مسیر را روشن می‌کنند, نه پایان را,  ما ادامه می‌دهیم, قوی‌تر, آگاه‌تر, زنده‌تر. با احترام علیرضا بهمنش راد  - @behmaneshrad"
    out = encode_lsb(input_image, secret_text, "encoded_image.png")
    print("Saved:", out)
